In [ ]:
# Selección lote de validación
!pip install xlsxwriter

import re, unicodedata
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from pandas import ExcelWriter

drive.mount('/content/drive')

# Paths
RUNS_DIR = "/content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/runs"
OUT_PATH = "/content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/lote_validacion_mistral.csv"
OUT_PATH_XLSX = "/content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/lote_validacion_mistral.xlsx"

# Parámetros de muestreo
N_PER_CAT = 10      # cantidad por categoría
RANDOM_SEED = 430
rng = np.random.RandomState(RANDOM_SEED)

# Normalización:
# NFC, colapso de espacios, remover signos de puntuación: { ", ", ., ,}
_final_punct_re = re.compile(r"[.,]+$")

def nfc(s: str) -> str:
    if s is None:
        return ""
    return unicodedata.normalize("NFC", str(s))

def strip_wrapping_quotes(s: str) -> str:
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        return s[1:-1]
    return s

def std_norm_min(s: str) -> str:
    s = nfc(s).strip()
    s = re.sub(r"\s+", " ", s)
    s = strip_wrapping_quotes(s)
    s = _final_punct_re.sub("", s)
    return s

# Cargar runs_all
runs_all_path = max(Path(RUNS_DIR).glob("runs_all_*.csv"), key=lambda p: p.stat().st_mtime)
df = pd.read_csv(runs_all_path)

# Normalización de pred y gold
df["pred_std"] = df["pred_quz"].fillna("").map(std_norm_min)
df["gold_std"] = df["gold_quz"].fillna("").map(std_norm_min)
view = df[df["pred_std"] == df["gold_std"]].copy()

# Deduplicar por id (ello en caso de que haya varias versiones del mismo id en runs_all)
if "id" in view.columns:
    view = view.drop_duplicates(subset=["id"], keep="first")

# muestreo balanceado por primary_change si es que existe, si no muestreo simple
if "primary_change" in view.columns:
    sampled = (
        view.groupby("primary_change", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), N_PER_CAT), random_state=rng))
    )
else:
    sampled = view.sample(min(len(view), N_PER_CAT * 5), random_state=rng)

# exportar columnas útiles
export_cols = [c for c in [
    "id","Source","Change","gold_quz","pred_quz",
    "primary_change","with_hints","K","latency_ms","split","size"
] if c in sampled.columns]

# Guardar CSV
sampled.to_csv(OUT_PATH, index=False, columns=export_cols)
print(f"Lote guardado en: {OUT_PATH}")
print("Total ejemplos:", len(sampled))

# Guardar XLSX
with ExcelWriter(OUT_PATH_XLSX, engine="xlsxwriter") as writer:
    sampled.to_excel(writer, sheet_name="Lote", index=False, columns=export_cols)
    ws = writer.sheets["Lote"]
    for i, col in enumerate(export_cols):
        width = max(12, min(60, sampled[col].astype(str).str.len().max() + 2))
        ws.set_column(i, i, width)
print(f"También guardado en: {OUT_PATH_XLSX}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.6 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lote guardado en: /content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/lote_validacion_mistral.csv
Total ejemplos: 73
También guardado en: /content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/lote_validacion_mistral.xlsx


/tmp/ipython-input-4157728096.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), N_PER_CAT), random_state=rng))
